In [5]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
persistent_directory = "db1/chroma_db"

# 1. Load Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Load DB
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}  
)

# 3. Retrieve Documents (Reducing k to 3 to save tokens on free tier)
query = "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
retriever = db.as_retriever(search_kwargs={"k": 3})
relevant_docs = retriever.invoke(query)

# 4. Initialize Gemini (Updated model name for 2026)
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0,
    max_retries=6,  # Automatically retries if you hit a rate limit
    convert_system_message_to_human=True # Better compatibility for Gemini
)

# 5. Prepare Input
combined_input = f"""Based on the following documents, answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in relevant_docs])}

If the answer isn't in the documents, say "I don't have enough information."
"""

messages = [
    SystemMessage(content="You are a helpful assistant specialized in Microsoft and GitHub history."),
    HumanMessage(content=combined_input),
]

# 6. Invoke and Print
try:
    result = model.invoke(messages)
    print("\n--- Generated Response (Gemini) ---")
    print('Answer: ',result.content)
except Exception as e:
    print(f"Error: {e}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2243.33it/s]



--- Generated Response (Gemini) ---
Answer:  The autonomous spaceport drone ship (ASDS) that achieved the first successful sea landing was named **Of Course I Still Love You**.
